# Phishing URL Detector (Educational Prototype)

**Project type:** Rule-based cybersecurity awareness tool  
**Purpose:** Inspect a URL and flag common warning signs that may be associated with phishing.

> **Important limitation:** This is a beginner-friendly educational prototype, not a production security product. It uses simple URL patterns and can produce false positives and false negatives. Never rely on it alone to decide whether a website is safe. Do not open suspicious links to test them.

## How it works
The program checks for several common warning signs:
- Use of an IP address instead of a domain name
- `@` symbol in the URL
- URL unusually long
- HTTPS missing
- Suspicious words such as `login`, `verify`, or `update`
- Many subdomains
- URL shortener domains

Each warning sign adds to a score. The final label is **Potentially Suspicious** or **No obvious warning signs detected**. The second label does **not** guarantee that a URL is safe.

In [ ]:
from urllib.parse import urlparse
import ipaddress

SUSPICIOUS_WORDS = {
    "login", "verify", "update", "secure", "account",
    "password", "signin", "banking", "confirm", "wallet"
}

SHORTENER_DOMAINS = {
    "bit.ly", "tinyurl.com", "t.co", "goo.gl", "ow.ly", "is.gd"
}

def normalize_url(url):
    """Add a scheme if the user enters a URL without http:// or https://."""
    url = url.strip()
    if not url:
        raise ValueError("Please enter a URL.")
    if "://" not in url:
        url = "http://" + url
    return url

def analyze_url(url):
    """Return a simple risk score, warning list, and educational label."""
    url = normalize_url(url)
    parsed = urlparse(url)
    hostname = (parsed.hostname or "").lower()
    full_url = url.lower()
    warnings = []
    score = 0

    if not hostname:
        return {
            "url": url,
            "score": 0,
            "label": "Invalid URL or missing hostname",
            "warnings": ["The URL could not be parsed into a hostname."]
        }

    # Check whether hostname is an IP address.
    try:
        ipaddress.ip_address(hostname)
        warnings.append("The hostname is an IP address rather than a regular domain.")
        score += 2
    except ValueError:
        pass

    if "@" in parsed.netloc:
        warnings.append("The URL contains an @ symbol, which can be used to mislead readers.")
        score += 2

    if len(url) > 75:
        warnings.append("The URL is unusually long.")
        score += 1

    if parsed.scheme != "https":
        warnings.append("The URL does not use HTTPS.")
        score += 1

    words_found = sorted(word for word in SUSPICIOUS_WORDS if word in full_url)
    if words_found:
        warnings.append("The URL contains words often used in account-related lures: " + ", ".join(words_found) + ".")
        score += 1

    # Count subdomains before the main domain, using a simple approximation.
    domain_parts = hostname.split(".")
    if len(domain_parts) >= 4:
        warnings.append("The hostname has many dot-separated parts (possible extra subdomains).")
        score += 1

    if hostname in SHORTENER_DOMAINS:
        warnings.append("The domain is a known URL-shortening service, which hides the destination.")
        score += 2

    if score >= 3:
        label = "Potentially Suspicious"
    else:
        label = "No obvious warning signs detected"

    if not warnings:
        warnings.append("No warning signs from this simple checklist were detected.")

    return {"url": url, "score": score, "label": label, "warnings": warnings}

print("URL checker is ready.")

## Try an example

In [ ]:
example_url = "http://192.0.2.10/login/verify-account"
result = analyze_url(example_url)

print("URL:", result["url"])
print("Risk score:", result["score"])
print("Result:", result["label"])
print("\nWarnings:")
for warning in result["warnings"]:
    print("-", warning)

## Check your own URL
Paste a URL into the cell below. Treat the result as a warning checklist only. Do not visit a suspicious link.

In [ ]:
url_to_check = input("Enter a URL to inspect (do not open it): ")
result = analyze_url(url_to_check)

print("\nURL:", result["url"])
print("Risk score:", result["score"])
print("Result:", result["label"])
print("\nWarnings:")
for warning in result["warnings"]:
    print("-", warning)

print("\nReminder: this educational tool cannot prove that a website is safe or malicious.")

## Limitations and future improvements
- This tool uses simple rules, not a trained machine-learning model.
- It does not check a website's reputation, certificate details, page content, or live threat feeds.
- Attackers can change URLs to avoid simple rules; legitimate sites can also trigger warnings.
- Future versions could use a labelled public dataset, evaluate precision/recall, and connect to a trusted threat-intelligence service with appropriate privacy and safety controls.